## Step 1: The Math Behind Batch Gradient Descent

Batch Gradient Descent is the standard, foundational form of the algorithm. The defining characteristic of "Batch" is that it uses the entire dataset to calculate the gradient (the slope of the error) for a single update step. If you have $N$ samples, the algorithm looks at all $N$ samples before taking one step downhill.

### 1. The Multi-Dimensional Hypothesis (Prediction)

For a dataset with $m$ features, our linear equation is an n-dimensional hyperplane:

$$\hat{y}_i = b + w_1x_{i1} + w_2x_{i2} + \dots + w_m x_{im}$$

We can write this cleanly in vector notation, where $W$ is the weight vector (coefficients) and $X_i$ is the feature vector for row $i$:

$$\hat{y}_i = X_i W + b$$

### 2. The Objective: Mean Squared Error (Loss Function)

Our goal is to minimize the average of all squared errors across all $N$ samples:

$$L(W, b) = \frac{1}{N} \sum_{i=1}^N (y_i - \hat{y}_i)^2$$

Substituting our hypothesis:

$$L(W, b) = \frac{1}{N} \sum_{i=1}^N (y_i - (X_i W + b))^2$$

### 3. Calculating the Gradients (Partial Derivatives)

To find out which way is downhill, we take the partial derivatives of the loss function with respect to our intercept ($b$) and our weight vector ($W$). Because we are using the entire batch, the summation runs over all $N$ samples.

**A. Derivative with respect to the Intercept ($b$):**

$$\frac{\partial L}{\partial b} = -\frac{2}{N} \sum_{i=1}^N (y_i - \hat{y}_i)$$

**B. Derivative with respect to the Weight vector ($W$):**

For any specific weight $w_j$ corresponding to feature $j$:

$$\frac{\partial L}{\partial w_j} = -\frac{2}{N} \sum_{i=1}^N x_{ij}(y_i - \hat{y}_i)$$

In vectorized form, the gradient for the entire weight matrix is calculated by multiplying the feature matrix $X^T$ by the error vector $(Y - \hat{Y})$:

$$\nabla_W L = -\frac{2}{N} X^T (Y - \hat{Y})$$

This matrix is very important it will give whole gradient at once .. i am using it in my custom class for batch gradient descent

### 4. The Update Rule

Once the gradients are computed over the entire batch, we update our parameters using the learning rate ($\eta$):

$$b_{new} = b_{old} - \eta \left( \frac{\partial L}{\partial b} \right)$$

$$W_{new} = W_{old} - \eta \left( \nabla_W L \right)$$

# Code Intuition separate class of this algortithm

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [4]:
diabetes = load_diabetes()
X = diabetes.data
y = diabetes.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# scikit learn benchmark (OLS method)
lr_sklearn = LinearRegression()
lr_sklearn.fit(X_train_scaled, y_train)
y_pred_sklearn = lr_sklearn.predict(X_test_scaled)
r2_sklearn = r2_score(y_test, y_pred_sklearn)

print("--- SCIKIT-LEARN OLS (THE TARGET) ---")
print(f"R2 Score:  {r2_sklearn:.6f}")
print(f"Intercept: {lr_sklearn.intercept_:.6f}")
print(f"All 10 Coefficients: {lr_sklearn.coef_}")

--- SCIKIT-LEARN OLS (THE TARGET) ---
R2 Score:  0.452603
Intercept: 153.736544
All 10 Coefficients: [  1.75375799 -11.51180908  25.60712144  16.82887167 -44.44885564
  24.64095356   7.67697768  13.1387839   35.16119521   2.35136365]


# Custom Batch Gradient Descent Class

In [7]:
class BatchGradientDescent:
  def __init__(self, learning_rate=0.1, epochs=1000):
    self.lr = learning_rate
    self.epochs = epochs
    self.coef_ = None
    self.intercept_ = None

  def fit(self, X_train, y_train):
    N, m = X_train.shape

    # init coeff(weights) and intercept(bias) to zero
    self.coef_ = np.zeros(m)  # because coef are exactly m which is columns here
    self.intercept_ = 0

    for i in range(self.epochs):
      # y_hat = X * W +b
      y_pred = np.dot(X_train, self.coef_) + self.intercept_

      # error = y_hat - y  here y_hat is y_pred and y is y_train
      error = y_pred - y_train

      # dl_db = (2/n) * summation of that error
      dl_db = (2/N) * np.sum(error)

      # now that coeff_matrix gradient: dot product of X_train transposed into error vecotr
      dl_dw = (2/N) * np.dot(X_train.T, error)

      # finally update rule
      self.intercept_ = self.intercept_ - (self.lr * dl_db)
      self.coef_ = self.coef_ - (self.lr * dl_dw)

  def predict(self, X_test):
      return np.dot(X_test, self.coef_) + self.intercept_

In [9]:
bgd = BatchGradientDescent(learning_rate=0.1, epochs=500)
bgd.fit(X_train_scaled, y_train)
y_pred_bgd = bgd.predict(X_test_scaled)
r2_bgd = r2_score(y_test, y_pred_bgd)

print("\n--- CUSTOM BATCH GRADIENT DESCENT ---")
print(f"R2 Score:  {r2_bgd:.6f}")
print(f"Intercept: {bgd.intercept_:.6f}")
print(f"All 10 Coefficients: {bgd.coef_}")


--- CUSTOM BATCH GRADIENT DESCENT ---
R2 Score:  0.454540
Intercept: 153.736544
All 10 Coefficients: [  1.8436046  -11.47718173  25.92462619  16.72667261 -27.95126816
  11.73638228   0.39863329  10.87259073  28.83044573   2.4708271 ]
